# AI Code Review Agent


In [ ]:
import sys

print("Installing packages... please wait 2-3 minutes...\n")

!{sys.executable} -m pip install -q langchain
!{sys.executable} -m pip install -q langchain-core
!{sys.executable} -m pip install -q langchain-groq
!{sys.executable} -m pip install -q streamlit
!{sys.executable} -m pip install -q python-dotenv

print("\n" + "="*50)
print(" SUCCESS — All packages installed!")
print("Run the next cell now.")
print("="*50)

Installing packages... please wait 2-3 minutes...




[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



✅ SUCCESS — All packages installed!
Run the next cell now.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Set Your Groq API Key


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


✅ API Key set successfully!


In [ ]:
print("Checking imports...\n")
failed = []

try:
    from langchain_groq import ChatGroq
    print(" ChatGroq")
except:
    print(" ChatGroq — run: pip install langchain-groq")
    failed.append("langchain-groq")

try:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    print(" ChatPromptTemplate + StrOutputParser")
except:
    print(" LangChain Core — run: pip install langchain-core")
    failed.append("langchain-core")

print("\n" + "="*50)
if failed:
    print(f" {len(failed)} failed. Run in Anaconda Prompt:")
    for p in failed:
        print(f"   pip install {p}")
    print("Then restart kernel and run from Cell 1")
else:
    print(" ALL IMPORTS OK — Continue!")
print("="*50)

Checking imports...

✅ ChatGroq
✅ ChatPromptTemplate + StrOutputParser

✅ ALL IMPORTS OK — Continue!


## Define the 4 Agent Tools


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def get_llm():
    """Returns Groq LLM (FREE)"""
    return ChatGroq(
        groq_api_key=os.environ["GROQ_API_KEY"],
        model_name="llama-3.3-70b-versatile",
        temperature=0.1,
        max_tokens=2048
    )

# ---- TOOL 1: BUG DETECTOR ----
def detect_bugs(code, language="Python"):
    print(" Tool 1: Bug Detection running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an expert code reviewer. "
         "Find ALL bugs in the code. "
         "For each bug: number, description, line, severity (CRITICAL/HIGH/MEDIUM/LOW), fix."
         "If no bugs say: No bugs found."),
        ("human", "Find bugs in this {language} code:\n```{language}\n{code}\n```")
    ])
    chain = prompt | get_llm() | StrOutputParser()
    return chain.invoke({"language": language, "code": code})

# ---- TOOL 2: SECURITY CHECKER ----
def check_security(code, language="Python"):
    print(" Tool 2: Security Analysis running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a cybersecurity expert. "
         "Find ALL security issues: SQL injection, hardcoded passwords, missing validation, etc. "
         "For each: name, risk (CRITICAL/HIGH/MEDIUM/LOW), what attacker can do, how to fix."
         "If no issues say: No security issues found."),
        ("human", "Security audit this {language} code:\n```{language}\n{code}\n```")
    ])
    chain = prompt | get_llm() | StrOutputParser()
    return chain.invoke({"language": language, "code": code})

# ---- TOOL 3: TEST GENERATOR ----
def generate_tests(code, language="Python"):
    print(" Tool 3: Unit Test Generation running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a senior software engineer. "
         "Write complete runnable unit tests. "
         "Use pytest for Python. Cover: normal cases, edge cases, error cases. "
         "Write real runnable code only."),
        ("human", "Write unit tests for this {language} code:\n```{language}\n{code}\n```")
    ])
    chain = prompt | get_llm() | StrOutputParser()
    return chain.invoke({"language": language, "code": code})

# ---- TOOL 4: DOC GENERATOR ----
def generate_docs(code, language="Python"):
    print(" Tool 4: Documentation Generation running...")
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a technical writer. "
         "Write professional documentation including: "
         "overview, each function with parameters/returns/examples, dependencies, usage example."),
        ("human", "Document this {language} code:\n```{language}\n{code}\n```")
    ])
    chain = prompt | get_llm() | StrOutputParser()
    return chain.invoke({"language": language, "code": code})

print(" All 4 tools defined!")
print("\nTools ready:")
print("   detect_bugs()")
print("   check_security()")
print("   generate_tests()")
print("   generate_docs()")

✅ All 4 tools defined!

Tools ready:
  🔍 detect_bugs()
  🔒 check_security()
  🧪 generate_tests()
  📝 generate_docs()


## Run Agent on Sample Code


In [ ]:
# Sample buggy code for testing
sample_code = """
import sqlite3

def get_user(username, password):
    conn   = sqlite3.connect('users.db')
    cursor = conn.cursor()
    # Bug 1: SQL Injection vulnerability!
    query  = "SELECT * FROM users WHERE username = '" + username + "'"
    cursor.execute(query)
    user   = cursor.fetchone()
    if user and user[2] == password:
        return user
    return None

def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    # Bug 2: Crashes if list is empty (ZeroDivisionError)!
    return total / len(numbers)

def save_config():
    # Bug 3: Hardcoded secret key!
    api_key = "sk-secret123456789"
    with open('config.txt', 'w') as f:
        f.write(api_key)
"""

LANGUAGE = "Python"

print(" AI Code Review Agent Starting...")
print(f"Language: {LANGUAGE}")
print(f"Code lines: {len(sample_code.strip().split(chr(10)))}")
print("Running all 4 tools...\n")

bug_result  = detect_bugs(sample_code, LANGUAGE)
sec_result  = check_security(sample_code, LANGUAGE)
test_result = generate_tests(sample_code, LANGUAGE)
docs_result = generate_docs(sample_code, LANGUAGE)

print("\n" + "="*55)
print(" ALL 4 TOOLS COMPLETE!")
print("Run next cells to see results.")
print("="*55)

🤖 AI Code Review Agent Starting...
Language: Python
Code lines: 25
Running all 4 tools...

🔍 Tool 1: Bug Detection running...
🔒 Tool 2: Security Analysis running...
🧪 Tool 3: Unit Test Generation running...
📝 Tool 4: Documentation Generation running...

✅ ALL 4 TOOLS COMPLETE!
Run next cells to see results.


In [ ]:
print("="*55)
print(" BUG REPORT")
print("="*55)
print(bug_result)

🔍 BUG REPORT
Here are the bugs found in the code:

1. **SQL Injection vulnerability**
   - Description: The `get_user` function is vulnerable to SQL injection attacks because it directly concatenates user input into the SQL query.
   - Line: 6
   - Severity: CRITICAL
   - Fix: Use parameterized queries or prepared statements to prevent SQL injection. For example:
     ```python
query  = "SELECT * FROM users WHERE username = ?"
cursor.execute(query, (username,))
```

2. **Division by zero error**
   - Description: The `calculate_average` function will crash with a ZeroDivisionError if the input list is empty.
   - Line: 14
   - Severity: HIGH
   - Fix: Add a check to handle the case where the input list is empty. For example:
     ```python
if len(numbers) == 0:
    return None  # or any other default value
return total / len(numbers)
```

3. **Hardcoded secret key**
   - Description: The `save_config` function uses a hardcoded secret key, which is a security risk.
   - Line: 18
   - Se

In [ ]:
print("="*55)
print(" SECURITY REPORT")
print("="*55)
print(sec_result)

🔒 SECURITY REPORT
### Security Audit Report

The provided Python code has several security issues that need to be addressed.

#### 1. SQL Injection Vulnerability
* **Name:** SQL Injection
* **Risk:** CRITICAL
* **What attacker can do:** An attacker can inject malicious SQL code to extract or modify sensitive data, potentially leading to unauthorized access, data breaches, or even complete control of the database.
* **How to fix:** Use parameterized queries or prepared statements to separate code from user input. In this case, use the `?` placeholder and pass the `username` as a parameter to `cursor.execute()`.
```python
query  = "SELECT * FROM users WHERE username = ?"
cursor.execute(query, (username,))
```

#### 2. Missing Input Validation
* **Name:** Missing Input Validation
* **Risk:** MEDIUM
* **What attacker can do:** An attacker can pass an empty list to the `calculate_average` function, causing a ZeroDivisionError. While not directly exploitable, this can lead to a denial-of-ser

## View Generated Unit Tests

In [ ]:
print("="*55)
print(" GENERATED UNIT TESTS")
print("="*55)
print(test_result)

🧪 GENERATED UNIT TESTS
```python
# tests.py
import pytest
import sqlite3
import os
from your_module import get_user, calculate_average, save_config  # Replace 'your_module' with the actual name of your module

# Create a test database
@pytest.fixture
def test_db():
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, password TEXT)')
    cursor.execute('INSERT INTO users (username, password) VALUES (?, ?)', ('test_user', 'test_password'))
    conn.commit()
    yield conn
    conn.close()

def test_get_user_normal_case(test_db):
    # Arrange
    test_db.cursor().execute('INSERT INTO users (username, password) VALUES (?, ?)', ('test_user', 'test_password'))
    test_db.commit()
    # Act
    user = get_user('test_user', 'test_password')
    # Assert
    assert user is not None
    assert user[1] == 'test_user'
    assert user[2] == 'test_password'

def test_get_user_edge_case_empty_username():
  

## View Generated Documentation

In [ ]:
print("="*55)
print(" GENERATED DOCUMENTATION")
print("="*55)
print(docs_result)

📝 GENERATED DOCUMENTATION
**Overview**
-----------

This Python module provides three functions: `get_user`, `calculate_average`, and `save_config`. The `get_user` function retrieves a user from a SQLite database based on their username and password. The `calculate_average` function calculates the average of a list of numbers. The `save_config` function saves a secret API key to a configuration file.

**Functions**
-------------

### get_user(username, password)

*   **Parameters:**
    *   `username` (str): The username of the user to retrieve.
    *   `password` (str): The password of the user to retrieve.
*   **Returns:**
    *   `user` (tuple or None): A tuple containing the user's data if found, otherwise None.
*   **Examples:**
    *   `get_user("john_doe", "password123")` returns the user's data if the username and password match.
*   **Note:**
    *   This function has a SQL injection vulnerability. It is recommended to use parameterized queries to prevent this.

### calculate_

In [20]:

my_code = """
def divide(a, b):
    return a / b

def find_max(items):
    max_val = items[0]
    for item in items:
        if item > max_val:
            max_val = item
    return max_val
"""

my_language = "Python"

# Run just bug detection on your code
# Change to check_security, generate_tests, or generate_docs as needed
result = detect_bugs(my_code, my_language)
print(result)

🔍 Tool 1: Bug Detection running...
Here are the bugs found in the code:

1. **Division by zero error**
   - Description: The `divide` function does not handle division by zero, which will raise a `ZeroDivisionError`.
   - Line: 2
   - Severity: CRITICAL
   - Fix: Add a check to raise a meaningful error or return a special value when `b` is zero. For example:
     ```python
def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b
```

2. **Index out of range error**
   - Description: The `find_max` function assumes that the input list `items` is not empty and tries to access the first element (`items[0]`). If the list is empty, this will raise an `IndexError`.
   - Line: 5
   - Severity: CRITICAL
   - Fix: Add a check to handle the case when the input list is empty. For example:
     ```python
def find_max(items):
    if not items:
        raise ValueError("Cannot find max of an empty list")
    max_val = items[0]
    for item in items:
       

In [ ]:
app_code = f'''import streamlit as st
import os, time
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["GROQ_API_KEY"] = "{os.environ.get('GROQ_API_KEY', '')}"

def get_llm():
    return ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"], model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=2048)

def run_tool(sys_msg, human_msg, code, language):
    prompt = ChatPromptTemplate.from_messages([("system", sys_msg),("human", human_msg)])
    return (prompt | get_llm() | StrOutputParser()).invoke({{"language": language, "code": code}})

st.set_page_config(page_title="AI Code Review Agent", page_icon="🤖", layout="wide")
st.title("🤖 AI Code Review Agent")
st.caption("Paste code → 4 AI tools analyze it → Get full review report")

with st.sidebar:
    language = st.selectbox("Language", ["Python","JavaScript","Java","TypeScript","C++"])
    st.divider()
    st.markdown("### Select Tools")
    do_bugs = st.checkbox(" Bug Detection",       value=True)
    do_sec  = st.checkbox(" Security Analysis",   value=True)
    do_test = st.checkbox(" Generate Unit Tests", value=True)
    do_docs = st.checkbox(" Generate Docs",       value=True)
    st.divider()
    st.markdown("**How it works:**")
    st.markdown("Code → Bug Tool → Security Tool → Test Tool → Doc Tool → Report")

if st.button(" Load Sample Code"):
    st.session_state["code"] = "import sqlite3\n\ndef get_user(username, password):\n    conn = sqlite3.connect(\'users.db\')\n    cursor = conn.cursor()\n    query = \'SELECT * FROM users WHERE username = \\\' + username + \'\\'\'\n    cursor.execute(query)\n    return cursor.fetchone()\n\ndef divide(a, b):\n    return a / b"

code = st.text_area(" Paste your code here:", value=st.session_state.get("code",""), height=280)
st.session_state["code"] = code

col1,col2,col3 = st.columns([2,1,2])
with col2:
    run = st.button(" Review Code", type="primary", use_container_width=True)

if run:
    if not code.strip(): st.error("Paste some code first!"); st.stop()
    if not any([do_bugs,do_sec,do_test,do_docs]): st.error("Select at least one tool!"); st.stop()
    results = {{}}
    elapsed = 0
    tools   = sum([do_bugs,do_sec,do_test,do_docs])
    done    = 0
    bar     = st.progress(0)
    status  = st.empty()
    start   = time.time()
    try:
        if do_bugs:
            status.text(" Running Bug Detection...")
            results[" Bugs"]     = run_tool("Find ALL bugs. For each: number, description, severity, fix.", "Find bugs in {{language}}:\n```{{language}}\n{{code}}\n```", code, language)
            done+=1; bar.progress(done/tools)
        if do_sec:
            status.text(" Running Security Analysis...")
            results[" Security"] = run_tool("Find ALL security issues: SQL injection, hardcoded passwords, etc. For each: name, risk, impact, fix.", "Security audit {{language}}:\n```{{language}}\n{{code}}\n```", code, language)
            done+=1; bar.progress(done/tools)
        if do_test:
            status.text(" Generating Unit Tests...")
            results[" Tests"]    = run_tool("Write complete runnable pytest unit tests. Cover normal, edge, error cases.", "Write tests for {{language}}:\n```{{language}}\n{{code}}\n```", code, language)
            done+=1; bar.progress(done/tools)
        if do_docs:
            status.text(" Generating Documentation...")
            results[" Docs"]     = run_tool("Write professional docs: overview, functions with params/returns/examples, usage.", "Document {{language}}:\n```{{language}}\n{{code}}\n```", code, language)
            done+=1; bar.progress(done/tools)
        elapsed = round(time.time()-start, 1)
        bar.progress(1.0)
        status.text(f" Complete in {{elapsed}}s!")
    except Exception as e:
        st.error(f"Error: {{e}}"); st.stop()
    st.markdown(f"###  Done — {{len(results)}} tools ran in {{elapsed}}s")
    tabs = st.tabs(list(results.keys()))
    for tab,(name,content) in zip(tabs, results.items()):
        with tab:
            st.markdown(content)
            st.download_button(f"⬇ Download", content, f"{{name.split()[1]}}_report.md", key=name)
    st.markdown("---")
    c1,c2,c3,c4 = st.columns(4)
    c1.metric("Tools Run",  len(results))
    c2.metric("Language",   language)
    c3.metric("Code Lines", len(code.split("\\n")))
    c4.metric("Time",       f"{{elapsed}}s")
'''

with open("agent_app.py", "w") as f:
    f.write(app_code)

print("✅ agent_app.py created successfully!")
print("\n" + "="*55)
print(" TO LAUNCH THE APP:")
print("  1. Open Anaconda Prompt")
print(f"  2. cd {os.getcwd()}")
print("  3. streamlit run agent_app.py")
print("  4. Browser opens at http://localhost:8501")
print("="*55)

UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f916' in position 744: character maps to <undefined>